In [1]:
import torch
import torch_directml
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import os

print("Step 1: Connecting to Radeon 9060 XT...")
device = torch_directml.device()
print(f"Hardware assigned: {device}")

print("\nStep 2: Loading Tokenized Dataset...")
dataset = torch.load("processed_data/tokenized_twitch_data.pt")

# Split the data into 80% train and 20% test
train_test_split = dataset["train"].train_test_split(test_size=0.2)
full_train_dataset = train_test_split["train"]
full_eval_dataset = train_test_split["test"]

# SPEED OPTIMIZATION: Slice a smaller subset for quick prototyping
# We select 5,000 rows for training and 1,000 rows for verification
train_dataset = full_train_dataset.select(range(5000))
eval_dataset = full_eval_dataset.select(range(1000))
print(f"Dataset optimized: Training on {len(train_dataset)} rows, evaluating on {len(eval_dataset)} rows.")

print("\nStep 3: Initializing DistilBERT Model...")
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", 
    num_labels=3
)

# Move the model to your AMD GPU
model.to(device)

print("\nStep 4: Setting Training Parameters...")
training_args = TrainingArguments(
    output_dir="./twitch_model_results",
    eval_strategy="epoch",          # Updated from 'evaluation_strategy' to fix the deprecation warning
    learning_rate=2e-5,             # Safe, optimal learning rate for transformers
    per_device_train_batch_size=32, # Bumped from 16 to 32 to maximize your 16GB VRAM throughput
    per_device_eval_batch_size=32,
    num_train_epochs=1,              # Reduced from 3 to 1 for a rapid verification run
    weight_decay=0.01,
    use_cpu=False,                   # Force GPU execution via DirectML
    save_strategy="no",              # Don't waste time saving massive checkpoint files mid-run
    report_to="none"                 # Disables external tracking warnings (Weights & Biases, etc.)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("\nStep 5: Beginning Training Loop! (This will be very fast)...")
trainer.train()

print("\nStep 6: Saving Trained Model...")
model.save_pretrained("./final_twitch_sentiment_model")
print("SUCCESS: Your model has a brain and is saved to disk!")

Step 1: Connecting to Radeon 9060 XT...
Hardware assigned: privateuseone:0

Step 2: Loading Tokenized Dataset...
Dataset optimized: Training on 5000 rows, evaluating on 1000 rows.

Step 3: Initializing DistilBERT Model...


C:\Users\ilyassbaa\AppData\Local\Temp\ipykernel_22716\202684097.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dataset = torch.load("processed_data/tokenized_twitch_da


Step 4: Setting Training Parameters...

Step 5: Beginning Training Loop! (This will be very fast)...


  0%|          | 0/157 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

{'eval_loss': 0.7693195939064026, 'eval_runtime': 19.7261, 'eval_samples_per_second': 50.694, 'eval_steps_per_second': 1.622, 'epoch': 1.0}
{'train_runtime': 406.6819, 'train_samples_per_second': 12.295, 'train_steps_per_second': 0.386, 'train_loss': 0.8692311086472432, 'epoch': 1.0}

Step 6: Saving Trained Model...
SUCCESS: Your model has a brain and is saved to disk!
